In [8]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 54.8 gigabytes of available RAM

You are using a high-RAM runtime!


In [9]:
!pip install xlsxwriter

In [10]:
!pip install spectral

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!ls "/content/drive/MyDrive/Cubos2/"

'Copy of raw_0_rd'	 HN2   LFH      LIMON2	 TEJON	   UNAM
'Copy of raw_0_rd.hdr'	 JMM   LIMON1   NAC2	 TEJON_2


In [13]:
"""
===============================================================================
EXPORTACIÓN DE CUBOS HIPERESPECTRALES NORMALIZADOS A EXCEL (.xlsx)
===============================================================================
Genera un archivo .xlsx por cada cubo hiperespectral.
Cada hoja del libro contiene los valores normalizados (0–1) de UNA banda,
preservando la estructura espacial 2D completa (filas × columnas del píxel).

VERSIÓN CORREGIDA: Usa archivos ENVI raw (raw_0_* + .hdr) vía librería
spectral, en lugar de GeoTIFF vía rasterio.

Requisitos:
    pip install xlsxwriter spectral numpy

Ejecución estimada: 15–40 min por cubo (depende de dimensiones y RAM de Colab).

Autor: Marcos — Tesis de Maestría, UNAM Facultad de Ciencias
Fecha: Febrero 2026
===============================================================================
"""

import os, time, numpy as np
from datetime import timedelta

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURACIÓN — Rutas a los archivos ENVI raw en Google Drive
# ─────────────────────────────────────────────────────────────────────────────

# Rutas a los archivos .hdr (spectral usa el .hdr para localizar el raw)
archivos_hdr = [
    '/content/drive/MyDrive/Cubos2/HN2/raw_0_HN2.hdr',
    '/content/drive/MyDrive/Cubos2/JMM/raw_0_JMM.hdr',
    '/content/drive/MyDrive/Cubos2/LFH/raw_0_LFH.hdr',
    '/content/drive/MyDrive/Cubos2/LIMON1/raw_0_LIMON_1.hdr',
    '/content/drive/MyDrive/Cubos2/LIMON2/raw_0_LIMON_2.hdr',
    '/content/drive/MyDrive/Cubos2/TEJON/raw_0_rd.hdr',
    '/content/drive/MyDrive/Cubos2/TEJON_2/raw_0_TEJON2.hdr',
    '/content/drive/MyDrive/Cubos2/NAC2/raw_0_NAC2.hdr',
    '/content/drive/MyDrive/Cubos2/UNAM/raw_0_UNAM.hdr',
]

# Nombres cortos para identificar cada cubo en los archivos de salida
nombres_cubos = [
    "HN2", "JMM1", "LFH2", "LIMON1_3",
    "LIMON2_4", "TEJON_5", "TEJON2_6", "NAC2", "UNAM"
]

# Directorio de salida en Google Drive
output_dir = "/content/drive/MyDrive/Cubos_Exportados_XLSX"

# Parámetros de recorte espacial (deben coincidir con tu pipeline existente)
ROW_RANGE = (0, 380)
COL_RANGE = (0, 400)

# ─────────────────────────────────────────────────────────────────────────────
# 2. FUNCIONES AUXILIARES
# ─────────────────────────────────────────────────────────────────────────────

def normalize_band(band):
    """Normaliza una banda 2D al rango [0, 1] usando min-max scaling."""
    min_val = np.nanmin(band)
    max_val = np.nanmax(band)
    if max_val - min_val == 0:
        return np.zeros_like(band, dtype=np.float32)
    return ((band - min_val) / (max_val - min_val)).astype(np.float32)


def cargar_y_recortar_cubo(hdr_path, row_range, col_range):
    """
    Carga un cubo hiperespectral ENVI (raw + .hdr) usando la librería spectral.

    NOTA IMPORTANTE sobre la forma del arreglo:
    - spectral devuelve shape (filas, columnas, bandas)
    - Lo transponemos a (bandas, filas, columnas) para mantener consistencia
      con el resto del pipeline de Marcos.

    Retorna: np.ndarray de shape (n_bandas, filas, columnas)
    """
    import spectral

    # spectral.open_image lee el .hdr y mapea el raw binario asociado
    img = spectral.open_image(hdr_path)
    print(f"Metadata ENVI: {img.shape[0]} filas × {img.shape[1]} cols × {img.shape[2]} bandas")

    # Cargar todo el cubo a memoria como NumPy array
    # .load() retorna shape (filas, columnas, bandas)
    data = img.load()

    # Aplicar recorte espacial
    r0, r1 = row_range
    c0, c1 = col_range
    r1 = min(r1, data.shape[0])
    c1 = min(c1, data.shape[1])
    data_recortado = np.array(data[r0:r1, c0:c1, :])

    # Transponer a (bandas, filas, columnas) para consistencia con el pipeline
    cubo = np.transpose(data_recortado, (2, 0, 1))

    return cubo


def exportar_cubo_a_xlsx(cubo, nombre_cubo, output_path):
    """
    Exporta un cubo hiperespectral normalizado a un archivo .xlsx.
    Cada banda se escribe en una hoja separada con nombre 'B000', 'B001', etc.

    Usa xlsxwriter con constant_memory=True para mínimo consumo de RAM.

    Parámetros:
        cubo        : np.ndarray de forma (n_bandas, filas, cols)
        nombre_cubo : str — identificador del cubo (e.g. "HN2")
        output_path : str — ruta completa del archivo .xlsx de salida
    """
    import xlsxwriter

    n_bandas, n_filas, n_cols = cubo.shape
    print(f"\n{'='*70}")
    print(f"  Exportando: {nombre_cubo}")
    print(f"  Dimensiones: {n_bandas} bandas × {n_filas} filas × {n_cols} columnas")
    print(f"  Celdas totales: {n_bandas * n_filas * n_cols:,.0f}")
    print(f"  Destino: {output_path}")
    print(f"{'='*70}")

    t_inicio = time.time()

    wb = xlsxwriter.Workbook(output_path, {'constant_memory': True})
    fmt_num = wb.add_format({'num_format': '0.0000'})
    fmt_header = wb.add_format({'bold': True, 'bg_color': '#D9E1F2', 'border': 1})

    for banda_idx in range(n_bandas):
        sheet_name = f"B{banda_idx:03d}"
        ws = wb.add_worksheet(sheet_name)

        # Normalizar esta banda
        banda_norm = normalize_band(cubo[banda_idx])

        # Encabezados de columna
        for col in range(n_cols):
            ws.write(0, col + 1, f"Col_{col}", fmt_header)

        # Encabezados de fila + datos
        for fila in range(n_filas):
            ws.write(fila + 1, 0, f"Fil_{fila}", fmt_header)
            ws.write_row(fila + 1, 1, banda_norm[fila].tolist(), fmt_num)

        # Progreso cada 25 bandas
        if (banda_idx + 1) % 25 == 0 or banda_idx == 0 or banda_idx == n_bandas - 1:
            elapsed = time.time() - t_inicio
            pct = (banda_idx + 1) / n_bandas * 100
            eta = timedelta(seconds=int(elapsed / (banda_idx + 1) * (n_bandas - banda_idx - 1)))
            print(f"  [{nombre_cubo}] Banda {banda_idx+1:3d}/{n_bandas} "
                  f"({pct:5.1f}%) — Transcurrido: {timedelta(seconds=int(elapsed))} — ETA: {eta}")

    wb.close()

    t_total = time.time() - t_inicio
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\n  ✓ {nombre_cubo} completado en {timedelta(seconds=int(t_total))}")
    print(f"  ✓ Tamaño del archivo: {size_mb:.1f} MB")
    return size_mb

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. EJECUCIÓN PRINCIPAL
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    os.makedirs(output_dir, exist_ok=True)

    n_cubos = len(archivos_hdr)

    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║  EXPORTACIÓN DE CUBOS HIPERESPECTRALES NORMALIZADOS A XLSX     ║")
    print(f"║  {n_cubos} cubos · ENVI raw · datos normalizados (0–1)                ║")
    print("╠══════════════════════════════════════════════════════════════════╣")
    print(f"║  Directorio de salida: {output_dir}")
    print(f"║  Recorte espacial: filas {ROW_RANGE}, columnas {COL_RANGE}")
    print("╚══════════════════════════════════════════════════════════════════╝\n")

    resultados = []
    t_global = time.time()

    for i, (hdr_path, nombre) in enumerate(zip(archivos_hdr, nombres_cubos)):
        print(f"\n[{i+1}/{n_cubos}] Cargando {nombre} desde: {hdr_path}")

        if not os.path.exists(hdr_path):
            print(f"  ⚠ ARCHIVO .hdr NO ENCONTRADO: {hdr_path} — Saltando...")
            resultados.append((nombre, "NO ENCONTRADO", 0, 0))
            continue

        # Verificar que el archivo raw binario asociado también existe
        raw_path = hdr_path.replace('.hdr', '')
        if not os.path.exists(raw_path):
            print(f"  ⚠ ARCHIVO raw NO ENCONTRADO: {raw_path} — Saltando...")
            resultados.append((nombre, "RAW NO ENCONTRADO", 0, 0))
            continue

        try:
            cubo = cargar_y_recortar_cubo(hdr_path, ROW_RANGE, COL_RANGE)
            print(f"  Cubo cargado y recortado: {cubo.shape}")

            if cubo.shape[0] < 10:
                print(f"  ⚠ ADVERTENCIA: {nombre} tiene solo {cubo.shape[0]} bandas.")

            output_path = os.path.join(output_dir, f"Cubo_{nombre}_normalizado.xlsx")
            size = exportar_cubo_a_xlsx(cubo, nombre, output_path)
            resultados.append((nombre, "OK", cubo.shape[0], size))

            # Liberar memoria
            del cubo

        except Exception as e:
            print(f"  ✗ ERROR procesando {nombre}: {e}")
            import traceback
            traceback.print_exc()
            resultados.append((nombre, f"ERROR: {e}", 0, 0))

    # ─── RESUMEN FINAL ───
    t_total_global = time.time() - t_global
    print("\n\n" + "="*70)
    print("  RESUMEN DE EXPORTACIÓN")
    print("="*70)
    print(f"  {'Cubo':<15} {'Estado':<20} {'Bandas':>8} {'Tamaño (MB)':>12}")
    print(f"  {'-'*15} {'-'*20} {'-'*8} {'-'*12}")
    total_size = 0
    for nombre, estado, bandas, size in resultados:
        print(f"  {nombre:<15} {estado:<20} {bandas:>8} {size:>11.1f}")
        total_size += size
    print(f"  {'-'*15} {'-'*20} {'-'*8} {'-'*12}")
    print(f"  {'TOTAL':<15} {'':<20} {'':>8} {total_size:>11.1f}")
    print(f"\n  Tiempo total: {timedelta(seconds=int(t_total_global))}")
    print(f"  Archivos guardados en: {output_dir}")
    print("="*70)

╔══════════════════════════════════════════════════════════════════╗
║  EXPORTACIÓN DE CUBOS HIPERESPECTRALES NORMALIZADOS A XLSX     ║
║  9 cubos · ENVI raw · datos normalizados (0–1)                ║
╠══════════════════════════════════════════════════════════════════╣
║  Directorio de salida: /content/drive/MyDrive/Cubos_Exportados_XLSX
║  Recorte espacial: filas (0, 380), columnas (0, 400)
╚══════════════════════════════════════════════════════════════════╝


[1/9] Cargando HN2 desde: /content/drive/MyDrive/Cubos2/HN2/raw_0_HN2.hdr
Metadata ENVI: 427 filas × 1004 cols × 325 bandas
  Cubo cargado y recortado: (325, 380, 400)

  Exportando: HN2
  Dimensiones: 325 bandas × 380 filas × 400 columnas
  Celdas totales: 49,400,000
  Destino: /content/drive/MyDrive/Cubos_Exportados_XLSX/Cubo_HN2_normalizado.xlsx
  [HN2] Banda   1/325 (  0.3%) — Transcurrido: 0:00:00 — ETA: 0:04:21
  [HN2] Banda  25/325 (  7.7%) — Transcurrido: 0:00:19 — ETA: 0:03:59
  [HN2] Banda  50/325 ( 15.4%) — Transcurr